# 1 - Config & Import

## 1.1 - Config

In [143]:
# 🔧 Config import
import os
from Project.Config.LoggerConfig import *
logger = colored_logger()
current_file = os.path.basename(__file__) if '__file__' in globals() else "Notebook"
logger.info(f"Logger initialized ({current_file})")

[2025-06-15 15:16:49] INFO - 1353130262.py - Logger initialized (Notebook)


## 1.2 - Import

In [222]:
# 📦 External libs
import plotly.graph_objects as go
import importlib
import torch
import torch.nn as nn
from torchinfo import summary
import torch.optim as optim

# 📂 Internal modules
import Data.DataFetcher
import Data.Features
import Data.Labels
import Data.Preprocessing
import Models.LSTMModel
import Data.DataAnalysis
import Trainer.LSTMTrainer

def reload_modules():
    importlib.reload(Data.DataFetcher)
    importlib.reload(Data.Features)
    importlib.reload(Data.Labels)
    importlib.reload(Data.DataAnalysis)
    importlib.reload(Data.Preprocessing)
    importlib.reload(Models.LSTMModel)
    importlib.reload(Trainer.LSTMTrainer)

# 2 - Data

## 2.1 - Data fetcher

### Variables

In [4]:
 #📊 Data parameters
symbol = "EURUSD"
start_date = "12-06-2025"
end_date = "13-06-2025"
interval = "1s"

### Process

In [6]:
from ib_insync import *
from datetime import datetime, timedelta
import pandas as pd
import asyncio
import random

ib = IB()

async def get_IB_FX_Data(symbol: str, start_date: datetime, end_date: datetime) -> pd.DataFrame:
    await ib.connectAsync('127.0.0.1', 4002, clientId=random.randint(100, 999))

    contract = Forex(symbol)
    data = []

    slice_duration = timedelta(minutes=1)
    total_requests = int((end_date - start_date) / slice_duration)
    print(f"Total requests to perform: {total_requests}")

    current_start = start_date

    for i in range(total_requests):
        current_end = min(current_start + slice_duration, end_date)

        try:
            ticks = await ib.reqHistoricalTicksAsync(
                contract,
                startDateTime=current_start,
                endDateTime=current_end,
                numberOfTicks=1000,
                whatToShow='BID_ASK',
                useRth=False
            )

            for tick in ticks:
                data.append({
                    'time': tick.time,
                    'bid': tick.priceBid,
                    'ask': tick.priceAsk,
                    'bidSize': tick.sizeBid,
                    'askSize': tick.sizeAsk
                })

        except Exception as e:
            print(f"Error fetching {symbol} from {current_start} to {current_end}: {e}")

        print(f"{i+1}/{total_requests}")
        current_start = current_end
        await asyncio.sleep(0.3)  # avoid pacing violation

    ib.disconnect()

    df = pd.DataFrame(data)
    return df


import nest_asyncio
nest_asyncio.apply()

start = datetime.utcnow() - timedelta(hours=1)
end = datetime.utcnow()

df = asyncio.run(get_IB_FX_Data(symbol, start, end))
print(df.head())


[2025-06-15 13:11:10] INFO - client.py - Connecting to 127.0.0.1:4002 with clientId 543...
[2025-06-15 13:11:10] INFO - client.py - Connected
[2025-06-15 13:11:10] INFO - client.py - Logged on to server version 176
[2025-06-15 13:11:10] INFO - client.py - API connection ready
[2025-06-15 13:11:10] INFO - wrapper.py - Warning 2104, reqId -1: Market data farm connection is OK:usfarm
[2025-06-15 13:11:10] INFO - wrapper.py - Warning 2106, reqId -1: HMDS data farm connection is OK:ushmds
[2025-06-15 13:11:10] INFO - wrapper.py - Warning 2158, reqId -1: Sec-def data farm connection is OK:secdefnj
[2025-06-15 13:11:10] INFO - ib.py - Synchronization complete


Total requests to perform: 60


[2025-06-15 13:11:12] INFO - wrapper.py - Warning 2106, reqId -1: HMDS data farm connection is OK:cashhmds


1/60
2/60
3/60
4/60
5/60
6/60
7/60
8/60
9/60
10/60
11/60
12/60
13/60
14/60
15/60
16/60
17/60
18/60
19/60
20/60
21/60
22/60
23/60
24/60
25/60
26/60
27/60
28/60
29/60
30/60
31/60
32/60
33/60
34/60
35/60
36/60
37/60
38/60
39/60
40/60
41/60
42/60
43/60
44/60
45/60
46/60
47/60
48/60
49/60
50/60
51/60
52/60
53/60
54/60
55/60
56/60
57/60
58/60
59/60
60/60


[2025-06-15 13:12:35] INFO - ib.py - Disconnecting from 127.0.0.1:4002, 6.59 kB sent in 68 messages, 12.4 kB received in 315 messages, session time 85.2 s.
[2025-06-15 13:12:35] INFO - client.py - Disconnecting


Empty DataFrame
Columns: []
Index: []


[2025-06-15 13:12:35] INFO - client.py - Disconnected.


In [223]:
reload_modules()

# 📥 Load data
fetcher = Data.DataFetcher.DataFetcher(symbol, start_date, end_date, interval)

#fetcher.get_binance_data()

#fetcher.save_to_csv(directory="./Data")

fetcher.load_from_csv_ib(directory="./Data")

fetcher.resample_to_1m_ohlcv()

[2025-06-15 17:24:17] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-15 17:24:17] INFO - Features.py - Logger initialized (Features.py)
[2025-06-15 17:24:17] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-15 17:24:17] INFO - DataAnalysis.py - Logger initialized (DataAnalysis.py)
[2025-06-15 17:24:17] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-15 17:24:17] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)
[2025-06-15 17:24:17] INFO - LSTMTrainer.py - Logger initialized (LSTMTrainer.py)
[2025-06-15 17:24:19] INFO - DataFetcher.py - 📥 Data loaded from: ./Data/EURUSD_12-06-2025_13-06-2025_1s.csv
[2025-06-15 17:24:20] INFO - DataFetcher.py - Resampling complete. Final shape: (1439, 7)


### Visualization

In [225]:
print("#-----------------------")
print(fetcher.raw_data.columns)
print("#-----------------------")
print(fetcher.raw_data.head())
print("#-----------------------")
print(fetcher.raw_data.shape)
print("#-----------------------")

size = 200
df = fetcher.raw_data.copy()[-size:]

# Create a candlestick chart using Plotly
fig = go.Figure(data=[
    go.Candlestick(
        x=df.index,  # Using the timestamp index directly
        open=df['Open'],
        high=df['High'],
        low=df['Low'],
        close=df['Close'],
        name="Candlesticks"
    )
])

# Customize layout
fig.update_layout(
    title=f"OHLC Candlestick Chart (Last {size} Points)",
    xaxis_title="Time",
    yaxis_title="Price",
    xaxis_rangeslider_visible=False,
    template="plotly_dark",
    height=500,
    width=int(500 * 1.618)
)

# Show the chart
fig.show()

#-----------------------
Index(['Open', 'High', 'Low', 'Close', 'Volume', 'bidSize', 'askSize'], dtype='object')
#-----------------------
                               Open      High       Low     Close    Volume  \
time                                                                          
2025-06-11 21:59:00+00:00  1.148820  1.148820  1.148820  1.148820    2000.0   
2025-06-11 22:00:00+00:00  1.148830  1.149160  1.148820  1.149125  396000.0   
2025-06-11 22:01:00+00:00  1.149140  1.149250  1.149130  1.149230  934000.0   
2025-06-11 22:02:00+00:00  1.149225  1.149255  1.149225  1.149250  498580.0   
2025-06-11 22:03:00+00:00  1.149250  1.149350  1.149240  1.149345  765220.0   

                            bidSize   askSize  
time                                           
2025-06-11 21:59:00+00:00    1000.0    1000.0  
2025-06-11 22:00:00+00:00  229000.0  167000.0  
2025-06-11 22:01:00+00:00  527000.0  407000.0  
2025-06-11 22:02:00+00:00  262880.0  235700.0  
2025-06-11 22:03:00+

## 2.2 - Feature Data

### Variables

In [226]:
# Resample with vwap
resample_period = '5min'

# Pivot Points
pivot_left = 15
pivot_right = 15

# Volume Pivot Points
duration_min = 240
n_cross = 7
std_factor = 1.0

# Std
std_window = 10

# Mean
mean_window = 10

### Process

In [227]:
reload_modules()

# Step 0: Initialize Features Dataframe
Features_Data = Data.Features.Features(fetcher.raw_data)

# Step 1: Resample with VWAP (Liquidity)
Features_Data.resample_with_vwap(resample_period)

# Step 2: Market Sessions (Market Context)
Features_Data.market_sessions()

# Step 3: Pivot Points (Support/Resistance)
Features_Data.Pivot_Points(pivot_left, pivot_right)

# Step 4: Volume Pivot Points (Volume-based S/R)
Features_Data.Volume_Pivot_Points(duration_min, n_cross, std_factor)

# Step 5: Volume Delta (Momentum)
Features_Data.add_volume_delta()

# Step 6: Cumulative Volume Delta - CVD (Momentum)
Features_Data.add_cvd()

# Step 7: Order Book Imbalance - OBI (Pressure)
Features_Data.add_obi()

# Step 8: Price Change (Price Action)
Features_Data.add_price_change()

# Step 9: Reaction Ratio (Liquidity)
Features_Data.add_reaction_ratio()

# Step 10: Rolling Standard Deviation of Price (Volatility)
Features_Data.add_rolling_std_price(std_window)

# Step 11: Rolling Mean of CVD (Smoothed Flow)
Features_Data.add_rolling_mean_cvd(mean_window)

[2025-06-15 17:24:33] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-15 17:24:33] INFO - Features.py - Logger initialized (Features.py)
[2025-06-15 17:24:33] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-15 17:24:33] INFO - DataAnalysis.py - Logger initialized (DataAnalysis.py)
[2025-06-15 17:24:33] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-15 17:24:33] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)
[2025-06-15 17:24:33] INFO - LSTMTrainer.py - Logger initialized (LSTMTrainer.py)
[2025-06-15 17:24:33] INFO - Features.py - Starting resampling with period: 5min
[2025-06-15 17:24:33] INFO - Features.py - Basic OHLCV resampling done.
[2025-06-15 17:24:33] INFO - Features.py - VWAP calculation completed.
[2025-06-15 17:24:33] INFO - Features.py - Resampling successful. Resulting shape: (293, 8)
[2025-06-15 17:24:33] INFO - Features.py - Starting market session flag generation.
[2025-06-15 17:24:33] INFO - Features.py

### Visualization

In [228]:
#-------------------------------------------------------------------------------

print("#----------------------------------------------------------------------")
print("Shape:", Features_Data.data.shape)
print("#----------------------------------------------------------------------")
print(Features_Data.data.columns)
print("#----------------------------------------------------------------------")
print(Features_Data.data.head())
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------
df = Features_Data.data.copy()[-300:]

# 1. Candlestick chart with VWAP_5m
fig1 = go.Figure()
fig1.add_trace(go.Candlestick(x=df.index,
                              open=df['Open'],
                              high=df['High'],
                              low=df['Low'],
                              close=df['Close'],
                              name="Candlestick"))
fig1.add_trace(go.Scatter(x=df.index, y=df['VWAP_5m'], mode='lines', name='VWAP_5m', line=dict(color='white')))
fig1.update_layout(title="Candlestick Chart with VWAP_5m", xaxis_title="Time", yaxis_title="Price", template="plotly_dark",xaxis_rangeslider_visible=False,)

# 2. Market session open status
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df.index, y=df["London_Open"], mode='lines', name="London"))
fig2.add_trace(go.Scatter(x=df.index, y=df["NY_Open"], mode='lines', name="New York"))
fig2.add_trace(go.Scatter(x=df.index, y=df["HK_Open"], mode='lines', name="Hong Kong"))
fig2.update_layout(title="Market Sessions", xaxis_title="Time", yaxis_title="Market Open (1=open)", template="plotly_dark")

# 3. Pivot points for price
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=df.index, y=df["Low"], mode='lines', name="Low", line=dict(width=0.8)))
fig3.add_trace(go.Scatter(x=df.index, y=df["Low_Pivot"], mode='lines', name="Low Pivot", line=dict(width=1.2, dash='dash')))
fig3.add_trace(go.Scatter(x=df.index, y=df["High"], mode='lines', name="High", line=dict(width=0.8)))
fig3.add_trace(go.Scatter(x=df.index, y=df["High_Pivot"], mode='lines', name="High Pivot", line=dict(width=1.2, dash='dash')))
fig3.update_layout(title="Price Pivot Points Detection", xaxis_title="Time", yaxis_title="Price", template="plotly_dark")

# 4. Volume pivot points
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=df.index, y=df["Volume_Low_Pivot"], mode='lines', name="Volume Low Pivot", line=dict(width=1.2, dash='dash')))
fig4.add_trace(go.Scatter(x=df.index, y=df["VWAP_5m"], mode='lines', name="VWAP 5m", line=dict(width=0.8)))
fig4.add_trace(go.Scatter(x=df.index, y=df["Rolling_VWAP_240min"], mode='lines', name="Rolling VWAP 240min"))
fig4.add_trace(go.Scatter(x=df.index, y=df["Volume_High_Pivot"], mode='lines', name="Volume High Pivot", line=dict(width=1.2, dash='dash')))
fig4.update_layout(title="Volume Pivot Points Detection", xaxis_title="Time", yaxis_title="Price", template="plotly_dark")

fig1.show()
fig2.show()
fig3.show()
fig4.show()

#----------------------------------------------------------------------
Shape: (293, 27)
#----------------------------------------------------------------------
Index(['Open', 'High', 'Low', 'Close', 'Volume', 'bidSize', 'askSize',
       'VWAP_5m', 'London_Open', 'NY_Open', 'HK_Open', 'Dif_Low_Pivot',
       'Dif_High_Pivot', 'Low_Pivot', 'High_Pivot', 'Rolling_VWAP_240min',
       'Volume_Low_Pivot', 'Volume_High_Pivot', 'Dif_Volume_Low_Pivot',
       'Dif_Volume_High_Pivot', 'volume_delta', 'CVD', 'obi', 'price_change',
       'reaction_ratio', 'rolling_std_price', 'rolling_mean_cvd'],
      dtype='object')
#----------------------------------------------------------------------
                         Open      High       Low     Close      Volume  \
time                                                                      
2025-06-11 21:55:00  1.148820  1.148820  1.148820  1.148820      2000.0   
2025-06-11 22:00:00  1.148830  1.149390  1.148820  1.149355   3265700.0   
2025-06-11

## 2.3 - Label Data

### Variables

In [229]:
# Step 1: Categorize Pivot Points
look_forward=20

# Step 2: Categorize Volume Pivot Points
look_forward=20

### Process

In [249]:
reload_modules()

# Step 0: Initialize Labels Dataframe
Labels_Data = Data.Labels.Labels(Features_Data.data)

# Step 1: Categorize Pivot Points
Labels_Data.categorize_pivot_points(look_forward)

# Step 2: Categorize Volume Pivot Points
Labels_Data.categorize_colume_pivot_points(look_forward)

[2025-06-15 17:35:01] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-15 17:35:01] INFO - Features.py - Logger initialized (Features.py)
[2025-06-15 17:35:01] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-15 17:35:01] INFO - DataAnalysis.py - Logger initialized (DataAnalysis.py)
[2025-06-15 17:35:01] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-15 17:35:01] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)
[2025-06-15 17:35:01] INFO - LSTMTrainer.py - Logger initialized (LSTMTrainer.py)
[2025-06-15 17:35:01] INFO - Labels.py - Starting Categorize_Pivot_Points with look_forward=20
[2025-06-15 17:35:01] INFO - Labels.py - Categorize_Pivot_Points completed successfully.
[2025-06-15 17:35:01] INFO - Labels.py - Starting Categorize_Volume_Pivot_Points with look_forward=20


<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.S

[2025-06-15 17:35:01] INFO - Labels.py - Categorize_Volume_Pivot_Points completed successfully.


### Visualization

In [231]:
#-------------------------------------------------------------------------------

print("#----------------------------------------------------------------------")
print("Shape:", Labels_Data.data.shape)
print("#----------------------------------------------------------------------")
print(Labels_Data.data.columns)
print("#----------------------------------------------------------------------")
print(Labels_Data.data.head())
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------

df = Labels_Data.data.copy()[-300:]

# Fréquences des 0 et 1 pour les colonnes cibles
frequency = df[["Low_Below_Pivot", "High_Above_Pivot"]].apply(lambda col: col.value_counts()).fillna(0).astype(int)
print("\nLabel Frequency (0 = no breakout, 1 = breakout):")
print(frequency.head(3))

# Initialiser les couleurs
colors_combined = ['lightgrey'] * len(df)
for i in range(len(df)):
    if df.iloc[i]['High_Above_Pivot'] == 1:
        colors_combined[i] = 'green'
    elif df.iloc[i]['Low_Below_Pivot'] == 1:
        colors_combined[i] = 'red'

# Création du graphique
fig = go.Figure()

# Trace VWAP_5m avec couleur dynamique
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['VWAP_5m'],
    mode='markers+lines',
    marker=dict(color=colors_combined, size=6),
    name='VWAP_5m',
    line=dict(color='lightgrey')
))

# Trace Low_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Low_Pivot'],
    mode='lines',
    line=dict(color='red', width=1),
    name='Low_Pivot'
))

# Trace High_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['High_Pivot'],
    mode='lines',
    line=dict(color='green', width=1),
    name='High_Pivot'
))

# Mise en page
fig.update_layout(
    title='VWAP_5m with Pivot Breakout Coloring and Pivot Lines',
    xaxis_title='Time',
    yaxis_title='Price',
    template='plotly_dark',
    height=500
)

fig.show()
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------

# Fréquences des 0 et 1 pour les colonnes VWAP
frequency = df[["VWAP_Below_Volume_Low", "VWAP_Above_Volume_High"]].apply(lambda col: col.value_counts()).fillna(0).astype(int)
print("\nLabel Frequency (0 = no breakout, 1 = breakout):")
print(frequency.head(3))

# Initialiser les couleurs
colors_combined = ['lightgrey'] * len(df)
for i in range(len(df)):
    if df.iloc[i]['VWAP_Above_Volume_High'] == 1:
        colors_combined[i] = 'green'
    elif df.iloc[i]['VWAP_Below_Volume_Low'] == 1:
        colors_combined[i] = 'red'

# Création du graphique
fig = go.Figure()

# Trace VWAP_5m avec couleur dynamique
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['VWAP_5m'],
    mode='markers+lines',
    marker=dict(color=colors_combined, size=6),
    name='VWAP_5m',
    line=dict(color='lightgrey')
))

# Trace Volume_Low_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Volume_Low_Pivot'],
    mode='lines',
    line=dict(color='red', width=1),
    name='Volume_Low_Pivot'
))

# Trace Volume_High_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Volume_High_Pivot'],
    mode='lines',
    line=dict(color='green', width=1),
    name='Volume_High_Pivot'
))

# Mise en page
fig.update_layout(
    title='VWAP_5m with Volume Pivot Breakout Coloring and Pivot Lines',
    xaxis_title='Time',
    yaxis_title='Price',
    template='plotly_dark',
    height=500
)

fig.show()

#-------------------------------------------------------------------------------

#----------------------------------------------------------------------
Shape: (293, 31)
#----------------------------------------------------------------------
Index(['Open', 'High', 'Low', 'Close', 'Volume', 'bidSize', 'askSize',
       'VWAP_5m', 'London_Open', 'NY_Open', 'HK_Open', 'Dif_Low_Pivot',
       'Dif_High_Pivot', 'Low_Pivot', 'High_Pivot', 'Rolling_VWAP_240min',
       'Volume_Low_Pivot', 'Volume_High_Pivot', 'Dif_Volume_Low_Pivot',
       'Dif_Volume_High_Pivot', 'volume_delta', 'CVD', 'obi', 'price_change',
       'reaction_ratio', 'rolling_std_price', 'rolling_mean_cvd',
       'Low_Below_Pivot', 'High_Above_Pivot', 'VWAP_Below_Volume_Low',
       'VWAP_Above_Volume_High'],
      dtype='object')
#----------------------------------------------------------------------
                         Open      High       Low     Close      Volume  \
time                                                                      
2025-06-11 21:55:00  1.148820  1.148820  1.148820  1.148

#----------------------------------------------------------------------

Label Frequency (0 = no breakout, 1 = breakout):
   VWAP_Below_Volume_Low  VWAP_Above_Volume_High
0                    264                     270
1                     29                      23


## 2.4 - Features/Labels Analysis

### Initilization

In [232]:
reload_modules()

print(Labels_Data.data.columns)

label_columns = ['Low_Below_Pivot', 'High_Above_Pivot', 'VWAP_Below_Volume_Low', 'VWAP_Above_Volume_High']

DataAnalysis_Data = Data.DataAnalysis.DataAnalysis(Labels_Data.data, label_columns)

[2025-06-15 17:24:43] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-15 17:24:43] INFO - Features.py - Logger initialized (Features.py)
[2025-06-15 17:24:43] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-15 17:24:43] INFO - DataAnalysis.py - Logger initialized (DataAnalysis.py)
[2025-06-15 17:24:43] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-15 17:24:43] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)
[2025-06-15 17:24:43] INFO - LSTMTrainer.py - Logger initialized (LSTMTrainer.py)


Index(['Open', 'High', 'Low', 'Close', 'Volume', 'bidSize', 'askSize',
       'VWAP_5m', 'London_Open', 'NY_Open', 'HK_Open', 'Dif_Low_Pivot',
       'Dif_High_Pivot', 'Low_Pivot', 'High_Pivot', 'Rolling_VWAP_240min',
       'Volume_Low_Pivot', 'Volume_High_Pivot', 'Dif_Volume_Low_Pivot',
       'Dif_Volume_High_Pivot', 'volume_delta', 'CVD', 'obi', 'price_change',
       'reaction_ratio', 'rolling_std_price', 'rolling_mean_cvd',
       'Low_Below_Pivot', 'High_Above_Pivot', 'VWAP_Below_Volume_Low',
       'VWAP_Above_Volume_High'],
      dtype='object')


### 1. Correlation Matrix

In [233]:
DataAnalysis_Data.plot_full_correlation_matrix()

### 2. Mutual Information (MI)

In [234]:
DataAnalysis_Data.plot_mutual_information_classification()

### 3. Feature Importance from Tree-Based Models

In [235]:
DataAnalysis_Data.plot_random_forest_importance()

### 4. Recursive Feature Elimination (RFE)

In [236]:
DataAnalysis_Data.plot_rfe_selected_features(n_features_to_select=10)


Full RFE ranking for target: Low_Below_Pivot
reaction_ratio            1
price_change              1
CVD                       1
obi                       1
volume_delta              1
Dif_Volume_High_Pivot     1
rolling_std_price         1
rolling_mean_cvd          1
Dif_Volume_Low_Pivot      1
Volume_High_Pivot         1
Volume_Low_Pivot          2
Rolling_VWAP_240min       3
High_Pivot                4
Low_Pivot                 5
Dif_High_Pivot            6
Dif_Low_Pivot             7
HK_Open                   8
NY_Open                   9
London_Open              10
VWAP_5m                  11
askSize                  12
bidSize                  13
Volume                   14
Close                    15
Low                      16
High                     17
Open                     18
dtype: int64



Full RFE ranking for target: High_Above_Pivot
Low                       1
VWAP_5m                   1
Rolling_VWAP_240min       1
Dif_Low_Pivot             1
Dif_High_Pivot            1
High_Pivot                1
CVD                       1
Dif_Volume_Low_Pivot      1
rolling_std_price         1
rolling_mean_cvd          1
Dif_Volume_High_Pivot     2
Volume_High_Pivot         3
High                      4
Close                     5
volume_delta              6
bidSize                   7
London_Open               8
askSize                   9
reaction_ratio           10
Volume                   11
Low_Pivot                12
Open                     13
obi                      14
price_change             15
NY_Open                  16
Volume_Low_Pivot         17
HK_Open                  18
dtype: int64



Full RFE ranking for target: VWAP_Below_Volume_Low
VWAP_5m                   1
Rolling_VWAP_240min       1
Dif_Low_Pivot             1
NY_Open                   1
CVD                       1
Dif_Volume_Low_Pivot      1
Volume_High_Pivot         1
Volume_Low_Pivot          1
rolling_std_price         1
rolling_mean_cvd          1
Open                      2
Dif_High_Pivot            3
High                      4
Low_Pivot                 5
Dif_Volume_High_Pivot     6
reaction_ratio            7
price_change              8
London_Open               9
Low                      10
Close                    11
volume_delta             12
askSize                  13
obi                      14
Volume                   15
HK_Open                  16
bidSize                  17
High_Pivot               18
dtype: int64



Full RFE ranking for target: VWAP_Above_Volume_High
High                      1
Low                       1
Rolling_VWAP_240min       1
Dif_Low_Pivot             1
High_Pivot                1
CVD                       1
Dif_Volume_Low_Pivot      1
Volume_High_Pivot         1
rolling_std_price         1
rolling_mean_cvd          1
Dif_Volume_High_Pivot     2
Dif_High_Pivot            3
Open                      4
VWAP_5m                   5
Close                     6
bidSize                   7
Volume                    8
London_Open               9
price_change             10
volume_delta             11
reaction_ratio           12
askSize                  13
obi                      14
NY_Open                  15
Low_Pivot                16
Volume_Low_Pivot         17
HK_Open                  18
dtype: int64


### 5. PCA (Principal Component Analysis)

In [237]:
DataAnalysis_Data.plot_pca_analysis()

### 6. Cross-Correlation (for time-series)

In [238]:
DataAnalysis_Data.plot_cross_correlation(max_lag=20)

C:\Users\valer\Desktop\Trading\Trading_Bot\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2999: RuntimeWarning:

invalid value encountered in divide

C:\Users\valer\Desktop\Trading\Trading_Bot\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3000: RuntimeWarning:

invalid value encountered in divide



C:\Users\valer\Desktop\Trading\Trading_Bot\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2999: RuntimeWarning:

invalid value encountered in divide

C:\Users\valer\Desktop\Trading\Trading_Bot\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3000: RuntimeWarning:

invalid value encountered in divide



C:\Users\valer\Desktop\Trading\Trading_Bot\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2999: RuntimeWarning:

invalid value encountered in divide

C:\Users\valer\Desktop\Trading\Trading_Bot\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3000: RuntimeWarning:

invalid value encountered in divide



C:\Users\valer\Desktop\Trading\Trading_Bot\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2999: RuntimeWarning:

invalid value encountered in divide

C:\Users\valer\Desktop\Trading\Trading_Bot\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3000: RuntimeWarning:

invalid value encountered in divide



### 7. Variance Inflation Factor (VIF) for Multicollinearity

In [239]:
DataAnalysis_Data.plot_vif_scores()


C:\Users\valer\Desktop\Trading\Trading_Bot\.venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning:

invalid value encountered in scalar divide



## 2.5  - Preprocessing

### Variables

In [240]:
# Step 0  Initialize Processing Data
print(Labels_Data.data.columns)
train_test_data = pd.DataFrame({"Feature_1": Labels_Data.data['VWAP_5m'],
                                "Feature_2": Labels_Data.data["London_Open"],
                                "Feature_3": Labels_Data.data["NY_Open"],
                                "Feature_4": Labels_Data.data["HK_Open"],
                                "Feature_5": Labels_Data.data['Dif_Low_Pivot'],
                                "Feature_6": Labels_Data.data['Dif_High_Pivot'],
                                "Feature_7": Labels_Data.data['Dif_Volume_Low_Pivot'],
                                "Feature_8": Labels_Data.data['Dif_Volume_High_Pivot'],

                                "Label_1": Labels_Data.data['Low_Below_Pivot'],
                                "Label_2": Labels_Data.data['High_Above_Pivot'],
                                "Label_3": Labels_Data.data['VWAP_Below_Volume_Low'],
                                "Label_4": Labels_Data.data['VWAP_Above_Volume_High'],
                                }).dropna()

# Step 1: Create Train/Test Data
lookback = 50
size_test_prct = 0.3

# Step 2: Suggest BatchSize
feature_dim = 9
label_dim = 4
lookback = 50
reserved_ram_gb = 1.0
hidden_dim = [64, 64]

# Step 3: Create DataLoaders
val_ratio = 0.2

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'bidSize', 'askSize',
       'VWAP_5m', 'London_Open', 'NY_Open', 'HK_Open', 'Dif_Low_Pivot',
       'Dif_High_Pivot', 'Low_Pivot', 'High_Pivot', 'Rolling_VWAP_240min',
       'Volume_Low_Pivot', 'Volume_High_Pivot', 'Dif_Volume_Low_Pivot',
       'Dif_Volume_High_Pivot', 'volume_delta', 'CVD', 'obi', 'price_change',
       'reaction_ratio', 'rolling_std_price', 'rolling_mean_cvd',
       'Low_Below_Pivot', 'High_Above_Pivot', 'VWAP_Below_Volume_Low',
       'VWAP_Above_Volume_High'],
      dtype='object')


### Process

In [241]:
# Step 0  Initialize Processing Data
Preprocessing_Data = Data.Preprocessing.Preprocessing(train_test_data)

# Step 1: Create Train/Test Data
Preprocessing_Data.create_train_test_data(lookback,size_test_prct)

# Step 2: Suggest BatchSize
Preprocessing_Data.suggest_batch_size(feature_dim, label_dim, lookback, reserved_ram_gb, hidden_dim)

# Step 3: Create DataLoaders
Preprocessing_Data.create_dataloaders(val_ratio)

[2025-06-15 17:25:01] INFO - Preprocessing.py - ✅ Using device: cpu
[2025-06-15 17:25:01] INFO - Preprocessing.py - Starting creation of train/test data (PyTorch).
[2025-06-15 17:25:01] INFO - Preprocessing.py - Selected 8 feature(s) and 4 label(s).
[2025-06-15 17:25:01] INFO - Preprocessing.py - Features normalized (min-max).
[2025-06-15 17:25:01] INFO - Preprocessing.py - Generated 50 sequences of length 50.
[2025-06-15 17:25:01] INFO - Preprocessing.py - Train/Test split: 35 train / 15 test samples.
[2025-06-15 17:25:01] INFO - Preprocessing.py - ✅ Train/test tensors successfully created (PyTorch).
[2025-06-15 17:25:01] INFO - Preprocessing.py - Starting batch size estimation (PyTorch)...
[2025-06-15 17:25:01] INFO - Preprocessing.py - Creating PyTorch DataLoaders...
[2025-06-15 17:25:01] INFO - Preprocessing.py - Train/Val split: 28 train / 7 val samples
[2025-06-15 17:25:01] INFO - Preprocessing.py - ✅ PyTorch DataLoaders created successfully.


In [242]:
pd.set_option('display.max_rows', None)     # Affiche toutes les lignes
pd.set_option('display.max_columns', None)  # Affiche toutes les colonnes
pd.set_option('display.width', 1000)        # Largeur max pour éviter le retour à la ligne
print(Preprocessing_Data.x_train)  # ou simplement train_test_data
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')

tensor([[[1.0000, 1.0000, 0.0000,  ..., 0.4230, 0.2514, 0.0109],
         [0.7529, 1.0000, 0.0000,  ..., 0.5992, 0.0000, 0.3262],
         [0.6012, 1.0000, 0.0000,  ..., 0.6984, 0.2742, 0.5197],
         ...,
         [0.3119, 0.0000, 1.0000,  ..., 0.8333, 0.4029, 0.2731],
         [0.3235, 0.0000, 1.0000,  ..., 0.8405, 0.4147, 0.0738],
         [0.3937, 0.0000, 1.0000,  ..., 0.8183, 0.4861, 0.1688]],

        [[0.7529, 1.0000, 0.0000,  ..., 0.5992, 0.0000, 0.3262],
         [0.6012, 1.0000, 0.0000,  ..., 0.6984, 0.2742, 0.5197],
         [0.6042, 1.0000, 0.0000,  ..., 0.6976, 0.2773, 0.5159],
         ...,
         [0.3235, 0.0000, 1.0000,  ..., 0.8405, 0.4147, 0.0738],
         [0.3937, 0.0000, 1.0000,  ..., 0.8183, 0.4861, 0.1688],
         [0.3721, 0.0000, 1.0000,  ..., 0.8246, 0.4642, 0.1963]],

        [[0.6012, 1.0000, 0.0000,  ..., 0.6984, 0.2742, 0.5197],
         [0.6042, 1.0000, 0.0000,  ..., 0.6976, 0.2773, 0.5159],
         [0.7059, 1.0000, 0.0000,  ..., 0.6270, 0.3807, 0.

### Verification

In [243]:
print("x_train shape:", Preprocessing_Data.x_train.shape)
print("y_train shape:", Preprocessing_Data.y_train.shape)
print("x_test shape:", Preprocessing_Data.x_test.shape)
print("y_test shape:", Preprocessing_Data.y_test.shape)

for row in Preprocessing_Data.report_lines:
    print(row)

for x_batch, y_batch in Preprocessing_Data.train_loader.take(1):
    print("Train batch shape:", x_batch.shape, y_batch.shape)

for x_batch, y_batch in Preprocessing_Data.val_loader.take(1):
    print("Val batch shape:", x_batch.shape, y_batch.shape)

x_train shape: torch.Size([35, 50, 8])
y_train shape: torch.Size([35, 4])
x_test shape: torch.Size([15, 50, 8])
y_test shape: torch.Size([15, 4])
--- Batch Size Estimation Report (PyTorch) ---
Total RAM (GB):         7.78
Reserved RAM (GB):      1.0
Usable RAM (GB):        6.78
Samples (n_samples):    35
Features per timestep:  9
Labels (n_labels):      4
Lookback (timesteps):   50
Hidden dimensions:      [64, 64] (layers: 2)
Bytes/sample:           51.77 KB
Max batch size (RAM):   137257.0
Max batch size (CPU):   64
Using GPU:              No
Final suggested batch:  32
--------------------------------------------------


AttributeError: 'DataLoader' object has no attribute 'take'

# 3 - Model

## 3.1 - Varibles

In [244]:
reload_modules()

# Set dimensions based on preprocessing
input_dim = Preprocessing_Data.x_train.shape[2]
output_dim = Preprocessing_Data.y_train.shape[1]
hidden_dim = 128  # Increased
num_layers = 2

dropout = 0.3  # Increased dropout

[2025-06-15 17:25:35] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-15 17:25:35] INFO - Features.py - Logger initialized (Features.py)
[2025-06-15 17:25:35] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-15 17:25:35] INFO - DataAnalysis.py - Logger initialized (DataAnalysis.py)
[2025-06-15 17:25:35] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-15 17:25:35] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)
[2025-06-15 17:25:35] INFO - LSTMTrainer.py - Logger initialized (LSTMTrainer.py)


## 3.2 - Model Initialization

In [245]:
# Initialize model
model = Models.LSTMModel.LSTMModel(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    num_layers=num_layers,
    dropout=dropout
).to(Preprocessing_Data.device)

# Model summary
print(summary(
    model,
    input_size=(Preprocessing_Data.batch_size, Preprocessing_Data.x_train.shape[1], input_dim),
    device=Preprocessing_Data.device
))

# Optional: pos_weight for imbalanced binary labels
total_positives = Preprocessing_Data.y_train.sum(dim=0)
total_negatives = Preprocessing_Data.y_train.shape[0] - total_positives
pos_weight = total_negatives / (total_positives + 1e-6)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(Preprocessing_Data.device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

Layer (type:depth-idx)                   Output Shape              Param #
LSTMModel                                [32, 4]                   --
├─LSTM: 1-1                              [32, 50, 128]             202,752
├─BatchNorm1d: 1-2                       [32, 128]                 256
├─Sequential: 1-3                        [32, 4]                   --
│    └─Linear: 2-1                       [32, 64]                  8,256
│    └─LeakyReLU: 2-2                    [32, 64]                  --
│    └─Dropout: 2-3                      [32, 64]                  --
│    └─Linear: 2-4                       [32, 4]                   260
Total params: 211,524
Trainable params: 211,524
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 324.68
Input size (MB): 0.05
Forward/backward pass size (MB): 1.69
Params size (MB): 0.85
Estimated Total Size (MB): 2.59


# 4 - Training & Visualization

## 4.1 - Variables

In [246]:
reload_modules()

[2025-06-15 17:25:41] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-15 17:25:41] INFO - Features.py - Logger initialized (Features.py)
[2025-06-15 17:25:41] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-15 17:25:41] INFO - DataAnalysis.py - Logger initialized (DataAnalysis.py)
[2025-06-15 17:25:41] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-15 17:25:41] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)
[2025-06-15 17:25:41] INFO - LSTMTrainer.py - Logger initialized (LSTMTrainer.py)


## 4.2 - Model Trainer

In [247]:
# Instantiate the trainer
trainer = Trainer.LSTMTrainer.LSTMTrainer(
    model=model,
    preprocessing=Preprocessing_Data,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    patience=5  # You can increase to 10 if needed
)

# Train the model
trainer.train(num_epochs=5)


Epoch 01 | Train Loss: 0.6457 | Val Loss: 0.6351 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000 | AUC: 0.6875 | ExactMatch: 0.2857 | HammingLoss: 0.2500
Epoch 02 | Train Loss: 0.6119 | Val Loss: 0.6335 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000 | AUC: 0.6875 | ExactMatch: 0.2857 | HammingLoss: 0.2500
Epoch 03 | Train Loss: 0.5885 | Val Loss: 0.6319 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000 | AUC: 0.6875 | ExactMatch: 0.2857 | HammingLoss: 0.2500
Epoch 04 | Train Loss: 0.5812 | Val Loss: 0.6304 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000 | AUC: 0.6875 | ExactMatch: 0.2857 | HammingLoss: 0.2500
Epoch 05 | Train Loss: 0.5242 | Val Loss: 0.6288 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000 | AUC: 0.7083 | ExactMatch: 0.2857 | HammingLoss: 0.2500


In [ ]:
model.load_state_dict(torch.load("best_model.pt"))
model.eval()
